# Turkey Business Activity — Admin notebook

**Read this only if you are the admin.** This notebook is not part of the normal
run flow — the cloud collector on the GCP VM does all the real work, 24/7.
Use this file when you need to:

- Verify the VM is publishing to Firestore right now (quick health check).
- Test a new camera before you add it to `app/cameras.py` GRID_SLOTS.
- Simulate a fallback swap to validate that the picker's chain is sensible.
- Run a manual **backup collector** locally if the VM is down for maintenance.

Zero-touch operation: if you never open this notebook, the system still runs.
The VM is the primary. Everything here is diagnostic.


## Setup

In [ ]:
%pip install -q ultralytics opencv-python-headless yt-dlp firebase-admin
import os, sys, time, datetime as dt
from pathlib import Path
import cv2, numpy as np

sys.path.append(str(Path.cwd()))
from app.cameras       import CAMERAS, GRID_SLOTS
from app.detect_core   import load_model, grab_frame, detect_with_boxes, resolve_stream
model = load_model('yolov8n.pt')
print('Grid slots:')
for s in GRID_SLOTS:
    chain = ' -> '.join([s['primary'], *s['fallbacks']])
    print(f'  {s["slot_id"]:20s} = {chain}')


## 1. VM health check — is the cloud collector actually writing?

Reads the four `latest/{slot_id}` docs from Firestore and reports the age of
each. If any slot is older than 2 minutes, the VM is either down, wedged, or
having stream issues — SSH to it and check `journalctl -u collector -f`.

You DO need a service-account JSON on this machine for this diagnostic.


In [ ]:
# Point at the local Firebase Admin key. Adjust the path if yours lives elsewhere.
_SA_CANDIDATES = ['serviceAccount.json',
                  '../serviceAccount.json',
                  'turkey-footfall-firebase-adminsdk-fbsvc-249068706b.json',
                  '../turkey-footfall-firebase-adminsdk-fbsvc-249068706b.json']
for _p in _SA_CANDIDATES:
    if Path(_p).is_file():
        os.environ['FIREBASE_CREDENTIALS'] = str(Path(_p).resolve())
        print(f'FIREBASE_CREDENTIALS -> {os.environ["FIREBASE_CREDENTIALS"]}')
        break
else:
    raise FileNotFoundError('No serviceAccount.json in the current or parent directory.')

from app.firebase_store import FirebaseStore
firebase = FirebaseStore()

now = dt.datetime.now(dt.timezone.utc)
print(f'\nSlot status (as of {now.isoformat(timespec="seconds")}):')
print('-' * 78)
for slot in GRID_SLOTS:
    doc = firebase.db.collection('latest').document(slot['slot_id']).get()
    if not doc.exists:
        print(f'  {slot["slot_id"]:20s} NO DATA (collector never wrote to this slot)')
        continue
    d = doc.to_dict()
    ts_iso = d.get('ts')
    age_s  = (now - dt.datetime.fromisoformat(ts_iso)).total_seconds() if ts_iso else None
    status = '✓ fresh' if age_s is not None and age_s < 120 else '! STALE'
    print(f'  {slot["slot_id"]:20s} {status:8s}  age={age_s:.0f}s  '
          f'cam={d.get("cam_id"):24s}  person={d.get("person")}  vehicles={d.get("vehicles")}')

# The current per-slot active cam according to the collector.
grid_doc = firebase.db.collection('config').document('grid').get()
if grid_doc.exists:
    g = grid_doc.to_dict()
    print(f'\nconfig/grid updated_at={g.get("updated_at")}')
    for s in g.get('slots', []):
        fallback = ' (FALLBACK)' if s.get('active_cam') != s.get('primary') else ''
        print(f'  {s["slot_id"]:20s} active={s.get("active_cam")}{fallback}')


## 2. Test a new camera before adding it to the catalog

Paste a candidate `cam` dict and this cell fetches a frame, runs YOLO, and
shows the annotated result — so you can decide whether it's worth adding.


In [ ]:
# Edit this dict to describe the new camera you want to try.
candidate = {
    'name': 'Candidate',
    'kind': 'hls',                                    # 'hls' | 'youtube' | 'skyline' | 'webcamera24'
    'url':  'https://example.tv/live/master.m3u8',    # source URL / page URL
    'page': 'https://example.tv/',                    # public webcam page (optional)
    'type': 'candidate',
}

try:
    stream_url = resolve_stream(candidate)
    frame = grab_frame(stream_url)
    if frame is None:
        print('WARN: no frame — the stream may be geo-blocked or down.')
    else:
        counts, _ = detect_with_boxes(model, frame, conf=0.3)
        print(f'frame: {frame.shape},  YOLO counts: {counts}')
        from app.detect_core import annotate
        import matplotlib.pyplot as plt
        plt.figure(figsize=(10, 6))
        plt.imshow(cv2.cvtColor(annotate(model, frame, conf=0.3), cv2.COLOR_BGR2RGB))
        plt.axis('off'); plt.title(candidate['name']); plt.show()
except Exception as e:
    print(f'FAIL: {e}')


## 3. Simulate a fallback swap

Exercise `SlotStreamPicker` locally to see whether the chain you defined in
`app/cameras.py` picks a sensible replacement when the primary fails. Nothing
touches Firestore — this is a dry run.


In [ ]:
from app.collector import SlotStreamPicker

for slot in GRID_SLOTS:
    p = SlotStreamPicker(slot)
    print(f'\n=== {slot["slot_id"]} ===')
    print(f'  start:            active = {p.current_cam()}')
    # Feed 6 straight failures — should advance twice (2 x MAX_FAILURES).
    for i in range(6):
        p.record_result(False)
    print(f'  after 6 misses:   active = {p.current_cam()}')
    p.record_result(True)
    print(f'  after 1 success:  active = {p.current_cam()}   (stays on current fallback)')


## 4. Backup collector (only when the VM is down)

If the VM is offline for maintenance / broken / geo-blocked, run this cell to
push samples yourself for a bounded window. It uses the SAME code paths as the
VM — including the fallback logic — so you're a drop-in replacement while it
lasts. Stops after `DURATION_MIN` minutes.


In [ ]:
# --- Backup collector (ADMIN ONLY, VM DOWN) --------------------------------
DURATION_MIN = 5
INTERVAL_S   = 20
CONF         = 0.30

from app.collector      import SlotStreamPicker, sample_slot, AnomalyTracker, \
                               FALLBACK_MAX_FAILURES, FALLBACK_RETRY_MINUTES
from app.firebase_store import FirebaseStore
from app.reid           import ReidStore

firebase = FirebaseStore()  # uses FIREBASE_CREDENTIALS from cell 1
print(f'Firebase Storage: {"ON" if firebase.storage else "OFF (local disk)"}')

pickers = {s['slot_id']: SlotStreamPicker(s) for s in GRID_SLOTS}
reid    = ReidStore('data/reid.db', threshold=0.92)
anomaly = AnomalyTracker()

# Publish current grid config so the dashboard knows what to render.
from app.collector import _slot_metadata
firebase.write_grid_config([_slot_metadata(s, pickers[s['slot_id']].current_cam())
                            for s in GRID_SLOTS])

end_at = time.time() + DURATION_MIN * 60
cycles = 0
try:
    while time.time() < end_at:
        for slot in GRID_SLOTS:
            picker = pickers[slot['slot_id']]
            cam_id = picker.current_cam()
            ok = sample_slot(model, slot, cam_id, firebase,
                             reid=reid, conf=CONF, anomaly=anomaly, save_snapshots=True)
            changed = picker.record_result(ok)
            if changed:
                firebase.write_grid_config([_slot_metadata(s, pickers[s['slot_id']].current_cam())
                                            for s in GRID_SLOTS])
                print(f'  * {slot["slot_id"]}: fallback -> {changed}')
        cycles += 1
        remaining = int(end_at - time.time())
        print(f'  cycle {cycles} done, {remaining}s remaining')
        time.sleep(max(0, INTERVAL_S - (time.time() % INTERVAL_S)))
    print(f'\n{DURATION_MIN}-min backup window finished; {cycles} cycles written.')
except KeyboardInterrupt:
    print(f'\nStopped early after {cycles} cycle(s).')
finally:
    reid.close()
